<div align="center">
<h1 style="background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%); color: white; padding: 30px; border-radius: 20px; margin-bottom: 20px;">🔐 KRİPTOQRAFİYA KURSU</h1>
<h2 style="color: #2c3e50; font-size: 2.5em; margin: 20px 0;">Məşğələ 28</h2>
<h2 style="color: #e74c3c; font-size: 2em; margin: 20px 0;">Post-Kvant Kriptoqrafiya I</h2>
<h3 style="color: #7f8c8d; font-size: 1.5em; margin: 20px 0;">Qəfəs Əsaslı Sistemlər</h3>
<br>
<div style="background-color: #3498db; color: white; padding: 15px; border-radius: 50px; display: inline-block; font-size: 1.2em;">⏱️ Məşğələ vaxtı: 2 saat</div>
<br><br>
<p><em>Hazırlanma tarixi: 2024 | Yenilənmiş versiya</em></p>
</div>

## 📑 Mündəricat

İcra müqaviləsi · quantum threat sərhədi · lattice/LWE toy problemləri · NIST PQC statusu · ML-KEM/ML-DSA ölçüləri · hybrid combiner · self-check və mənbələr


## 🎯 Məqsədlər

Shor və Grover təsirini qarışdırmamaq; quantum timeline proqnozu verməmək; lattice/CVP və LWE toy modelini exhaustive yoxlamaq; KEM/signature kateqoriyalarını ayırmaq; FIPS 203/204/205 və inkişafda olan FIPS 206/HQC statusunu düzgün göstərmək; hybrid combiner-in profile-specific olduğunu izah etmək.


## 📚 İcra mühiti

Standard-library-only notebook. `liboqs` native dependency, paket quraşdırılması və hidden fallback yoxdur. Toy kod ML-KEM, ML-DSA və ya quantum computer implementation deyil.


In [ ]:
import sys
print(f"Standard-library-only: Python {sys.version_info.major}.{sys.version_info.minor}")


## 🔧 Hazırlıq

Hər code cell yuxarıdan aşağıya işləyir. Standart ölçüləri metadata kimi yoxlanır; kriptoqrafik primitive iddiası yalnız standartlaşdırılmış API/implementation ilə edilə bilər.


### Termin müqaviləsi

KEM `KeyGen/Encaps/Decaps`, signature isə `KeyGen/Sign/Verify` interfeysidir. KEM ciphertext-i signature ilə ölçü cədvəlində eyni obyekt kimi müqayisə edilmir.


In [ ]:
import hashlib
import hmac
import math
import random
import secrets
import statistics

RNG=random.Random(30)
assert hashlib.sha256 and hmac.new
print('Məcburi API-lər hazırdır')


### Sübut sərhədi

Toy LWE secret recovery kiçik parametrin zəifliyini göstərir, ML-KEM təhlükəsizliyi haqqında nəticə vermir. Published byte sizes performans nəticəsi deyil.


In [ ]:
LAB_CONTRACT={'native_oqs':False,'network':False,'toy_is_pqc':False,'status_date':'2026-07-21','mandatory_cells':True}
assert not any(LAB_CONTRACT[k] for k in ('native_oqs','network','toy_is_pqc'))
assert LAB_CONTRACT['mandatory_cells']
print(LAB_CONTRACT)


## 📖 Quantum threat

Shor kifayət qədər böyük fault-tolerant quantum computer-də factoring və discrete logarithm üçün polynomial-time alqoritmdir; RSA/DH/ECC riskinə səbəb olur. Grover ideal black-box search-də quadratic speedup verir; symmetric/hash parameter seçimi və real resource cost ayrıca qiymətləndirilir. RSA-2048 üçün uydurma `2^32` quantum complexity verilmir.


### Timeline qeyri-müəyyəndir

Kriptoqrafik cəhətdən relevant fault-tolerant cihazın tarixi məlum fakt deyil. Migration qərarı inventory, data lifetime (“harvest now, decrypt later”), standards, vendor readiness və transition cost ilə verilir.


## ⚛️ NISQ və fault tolerance

NISQ cihazları noise və məhdud error correction ilə işləyir. Logical qubit, physical qubit, gate fidelity, error correction və algorithm resource estimate fərqli ölçülərdir; sadə qubit sayı threat tarixi vermir.


## 🚀 Post-kvant kriptoqrafiya

PQC klassik kompüterlərdə işləyən və məlum klassik/quantum hücumlara qarşı qiymətləndirilən alqoritmlərdir. “Quantum-resistant” gələcək bütün cryptanalysis-ə zəmanət deyil; implementation, protocol və key management qalır.


### Ailələr

Structured lattice (ML-KEM/ML-DSA), hash-based (SLH-DSA), lattice trapdoor (FN-DSA development), code-based (HQC selected), eləcə də əlavə signature candidates. Bir ailənin zəifliyi digərlərinə avtomatik keçmir.


### 5.1 NIST statusu — 2026-07-21

- FIPS 203 ML-KEM, FIPS 204 ML-DSA, FIPS 205 SLH-DSA: 13 avqust 2024 final.
- FN-DSA/FALCON: FIPS 206 inkişaf mərhələsində; FIPS 205 deyil.
- HQC: 2025-də əlavə KEM kimi seçilib, standart inkişaf mərhələsində.
- 9 əlavə signature candidate 2026 mayında üçüncü round-a keçib.

Status date həmişə mənbə ilə saxlanır.


## 🔷 Lattice əsasları

Lattice integer basis kombinasiyalarının diskret çoxluğudur. Eyni lattice müxtəlif basis-lərə malik ola bilər; basis keyfiyyəti hesablama çətinliyinə təsir edə bilər.


### 6.1 2D toy lattice

Kod yaxın lattice point-i məhdud coefficient pəncərəsində exhaustive axtarır; bu CVP solver və ya security estimate deyil.


### 6.2 SVP və CVP

SVP sıfırdan fərqli ən qısa lattice vector-u, CVP target-ə ən yaxın lattice vector-u axtarır. Approximation factor, norm, dimension və problem variantı göstərilmədən “SVP çətindir” kifayət deyil.


In [ ]:
def lattice_point(basis,coeffs): return tuple(sum(c*basis[j][i] for j,c in enumerate(coeffs)) for i in range(len(basis[0])))
def bounded_cvp(basis,target,bound=5):
    best=None
    for a in range(-bound,bound+1):
      for b in range(-bound,bound+1):
       point=lattice_point(basis,(a,b)); d2=sum((x-y)**2 for x,y in zip(point,target))
       candidate=(d2,(a,b),point)
       if best is None or candidate<best: best=candidate
    return best
basis=((2,0),(1,3)); cvp_result=bounded_cvp(basis,(5,5))
assert cvp_result==(2,(1,2),(4,6))
print({'distance_squared':cvp_result[0],'coefficients':cvp_result[1],'bounded_toy':True})


## 🔍 Lattice attacks

Enumeration, sieving və basis reduction (LLL/BKZ) müxtəlif cost/quality tradeoff-larına malikdir. Security estimate dimension, modulus, secret/error distribution, module/ring structure və attack model tələb edir; 2D şəkil estimate deyil.


### ✍️ Çalışma 1 — 1 bal

Üç target və iki search bound üçün bounded CVP-ni brute-force oracle ilə yoxlayın. Meyar: exact distance, boundary warning və production security claim yoxdur.


## 📊 LWE

LWE sample `b = <a,s> + e mod q` formasındadır. Secret və error distribution problemin parametridir. Error olmadan sistem modular linear equations-ə çevrilir; çox kiçik error/parameter brute-force recovery-ni asanlaşdıra bilər.


### 7.1 Toy LWE recovery

Kiçik `q=17,n=2` secret bütün 289 variant üzrə axtarılır. Candidate hər sample residual-ının icazəli error set-də olmasını tələb edir.


In [ ]:
def lwe_samples(secret,q,count):
 samples=[]
 for i in range(count):
  a=((3*i+1)%q,(5*i+2)%q); e=(-1,0,1)[i%3]
  b=(sum(x*y for x,y in zip(a,secret))+e)%q; samples.append((a,b))
 return samples
def recover_toy_lwe(samples,q,errors=(-1,0,1)):
 allowed={e%q for e in errors}; out=[]
 for x in range(q):
  for y in range(q):
   if all((b-(a[0]*x+a[1]*y))%q in allowed for a,b in samples): out.append((x,y))
 return out
lwe_secret=(4,9); lwe_data=lwe_samples(lwe_secret,17,24); lwe_candidates=recover_toy_lwe(lwe_data,17)
assert lwe_candidates==[lwe_secret]
print({'candidates_tested':289,'samples':24,'recovered_toy_secret':True})


## 📊 Error distribution

ML-KEM/ML-DSA konkret centered binomial/rejection-sampling qaydalarından istifadə edir; generic floating Gaussian demo onların sampler-i deyil. Aşağıda yalnız centered-binomial toy distribution göstərilir.


In [ ]:
def centered_binomial(eta,rng): return sum(rng.getrandbits(1) for _ in range(eta))-sum(rng.getrandbits(1) for _ in range(eta))
cb_samples=[centered_binomial(3,RNG) for _ in range(6000)]
assert min(cb_samples)>=-3 and max(cb_samples)<=3
assert abs(statistics.mean(cb_samples))<0.1
print({'samples':len(cb_samples),'range':(min(cb_samples),max(cb_samples)),'mean':round(statistics.mean(cb_samples),3)})


## 🔄 Ring-LWE və Module-LWE

Ring/module structure efficiency verir, lakin plain LWE vector-u sadəcə ad dəyişməklə Module-LWE olmur. Polynomial ring, module rank, modulus və distributions konkret scheme profilindən götürülür.


## 🔷 NTRU

NTRU ailəsi polynomial rings və fərqli hard-problem/decoding structure istifadə edir. “Bütün lattice schemes LWE-dir” doğru deyil.


### ✍️ Çalışma 2 — 1 bal

Toy LWE üçün sample count/error set dəyişdirin. Meyar: ambiguity/no-solution cases və ML-KEM-ə security extrapolation yoxdur.


## 🔵 ML-KEM (FIPS 203)

ML-KEM standardized KEM-dir: key generation, encapsulation və decapsulation shared secret yaradır. “Kyber encryption” termini KEM contract-ını gizlədə bilər. Parameter sets: ML-KEM-512/768/1024.


### 8.1 Status və istifadə

FIPS 203 finaldır. Parameter set seçimi yalnız category nömrəsi ilə deyil, protocol profile, key/ciphertext ölçüsü, platforma və compliance ilə edilir.


### 8.2 Decapsulation sərhədi

KEM failure behavior və implicit rejection protocol security üçün vacibdir. Toy hash funksiyası ML-KEM decapsulation-u simulyasiya etmir.


In [ ]:
ML_KEM_SIZES={
'ML-KEM-512':{'ek':800,'dk':1632,'ct':768,'ss':32,'category':1},
'ML-KEM-768':{'ek':1184,'dk':2400,'ct':1088,'ss':32,'category':3},
'ML-KEM-1024':{'ek':1568,'dk':3168,'ct':1568,'ss':32,'category':5}}
assert [v['category'] for v in ML_KEM_SIZES.values()]==[1,3,5]
assert all(v['ss']==32 for v in ML_KEM_SIZES.values())
print(ML_KEM_SIZES)


In [ ]:
def compare_same_artifact(table,field): return {name:values[field] for name,values in table.items()}
mlkem_ciphertext_sizes=compare_same_artifact(ML_KEM_SIZES,'ct')
assert mlkem_ciphertext_sizes=={'ML-KEM-512':768,'ML-KEM-768':1088,'ML-KEM-1024':1568}
print({'artifact':'KEM ciphertext bytes','values':mlkem_ciphertext_sizes})


### ✍️ Çalışma 3 — 1 bal

FIPS 203-dən ek/dk/ct/ss cədvəlini yenidən yoxlayın. Meyar: eyni artifact müqayisəsi, source table reference və benchmark claim yoxdur.


## 🟢 ML-DSA (FIPS 204)

ML-DSA standardized signature scheme-dir; parameter sets ML-DSA-44/65/87. KEM shared secret/ciphertext sahələri ilə signature size bir sütunda müqayisə edilmir.


### 9.1 Signature contract

KeyGen/Sign/Verify, context/string və randomized/deterministic variants konkret FIPS 204 profile ilə tətbiq olunur. HMAC və ya hash toy kodu ML-DSA deyil.


### 9.2 Ölçülər

Public/private key və signature encodings ayrı artifact-lardır. Implementation memory və wire framing bu raw byte counts-a əlavə ola bilər.


In [ ]:
ML_DSA_SIZES={
'ML-DSA-44':{'pk':1312,'sk':2560,'sig':2420,'category':2},
'ML-DSA-65':{'pk':1952,'sk':4032,'sig':3309,'category':3},
'ML-DSA-87':{'pk':2592,'sk':4896,'sig':4627,'category':5}}
assert [v['category'] for v in ML_DSA_SIZES.values()]==[2,3,5]
assert all(v['sig']>v['pk'] for v in ML_DSA_SIZES.values())
print(ML_DSA_SIZES)


## 🟠 FN-DSA/FALCON

FALCON əsasında FN-DSA FIPS 206 üçün inkişaf mərhələsindədir (2026-07-21 statusu). O, FIPS 205 deyil; FIPS 205 SLH-DSA-dır. Draft/final status source və date ilə göstərilməlidir.


### ✍️ Çalışma 4 — 1 bal

ML-DSA və SLH-DSA-nı signature-to-signature müqayisə edin. Meyar: exact parameter set/artifact, final standards və platformadan asılı timing ayrı saxlanır.


## 📊 Klassik sistemlərlə müqayisə

KEM-i RSA encryption, signature-i RSA/ECDSA signature ilə yalnız eyni protocol goal və artifact üzrə müqayisə edin. Shor polynomial-time threat-dir; uydurma “RSA quantum 2^32” sətri istifadə olunmur.


In [ ]:
pqc_artifact_tables={'KEM ciphertext':mlkem_ciphertext_sizes,'signature':compare_same_artifact(ML_DSA_SIZES,'sig')}
assert set(pqc_artifact_tables)=={'KEM ciphertext','signature'}
assert not (set(pqc_artifact_tables['KEM ciphertext']) & set(pqc_artifact_tables['signature']))
print(pqc_artifact_tables)


## 🛡️ NIST categories

Categories 1–5 benchmark reference attacks və resource metrics əsasında müqayisə çərçivəsidir; “dəqiq 128/192/256-bit security” kimi bərabərlik deyil. Concrete security attack model və parameters-dan asılıdır.


## 🔄 Hybrid yanaşma

Hybrid key establishment klassik və PQ components-i profile-defined combiner ilə birləşdirə bilər. Sadə concatenation və ya “biri təhlükəsizdirsə həmişə təhlükəsiz” universal theorem deyil; transcript binding, authentication, downgrade və failure behavior vacibdir.


In [ ]:
def hkdf_extract(salt,ikm): return hmac.new(salt,ikm,hashlib.sha256).digest()
def hkdf_expand(prk,info,length=32):
 out=b'';t=b'';counter=1
 while len(out)<length:
  t=hmac.new(prk,t+info+bytes([counter]),hashlib.sha256).digest();out+=t;counter+=1
 return out[:length]
def demo_hybrid_combiner(classical,pq,transcript): return hkdf_expand(hkdf_extract(bytes(32),classical+pq),b'AS28 hybrid|'+transcript)
c_secret=secrets.token_bytes(32);p_secret=secrets.token_bytes(32)
hybrid_key=demo_hybrid_combiner(c_secret,p_secret,b'profile-v1')
assert len(hybrid_key)==32
assert hybrid_key!=demo_hybrid_combiner(c_secret,bytes(32),b'profile-v1')
assert hybrid_key!=demo_hybrid_combiner(c_secret,p_secret,b'profile-v2')
print({'derived_bytes':32,'component_and_transcript_sensitive':True,'standard_profile_claim':False})


## 🌐 TLS inteqrasiyası

TLS hybrid groups və identifiers yalnız konkret IETF specification və implementation support ilə istifadə olunur. Notebook wire code yaratmır və browser/server support-u uydurmur.


### ✍️ Çalışma 5 — 1 bal

Bir hybrid draft/profile üçün identifiers, combiner, transcript, downgrade və failure behavior yazın. Meyar: source/date və toy HKDF-in həmin profile implementation olmadığı qeyd olunur.


## 🖥️ Self-check

Self-check toy math və metadata consistency-ni yoxlayır, PQ primitive-i yox.


In [ ]:
def run_pqc_self_check():
 checks={'cvp':cvp_result==(2,(1,2),(4,6)),'lwe':lwe_candidates==[lwe_secret],'sampler':abs(statistics.mean(cb_samples))<0.1,
 'mlkem':len(ML_KEM_SIZES)==3,'mldsa':len(ML_DSA_SIZES)==3,'same_artifact':len(pqc_artifact_tables)==2,
 'hybrid':len(hybrid_key)==32,'no_native_oqs':not LAB_CONTRACT['native_oqs']}
 assert all(checks.values());return checks
pqc_checks=run_pqc_self_check();print({'passed':sum(pqc_checks.values()),'total':len(pqc_checks)})


## 🏠 Ev tapşırığı — 10 bal

FIPS 203/204 migration note: inventory 2; exact artifacts 2; protocol profile 2; hybrid/downgrade 1; negative tests 1; key lifecycle 1; dated primary sources/limitations 1. Native library məcburi deyil və toy code standardized primitive adlandırılmır.


## 📌 Yekun

FIPS 203=ML-KEM, 204=ML-DSA, 205=SLH-DSA; FIPS 206/FN-DSA inkişafda. Categories exact bit equality deyil. KEM və signature artifacts ayrıdır. Quantum deadline məlum deyil. Hybrid profile-specific-dir. Toy LWE/Python hash PQC implementation deyil.


## 📚 Mənbələr

- NIST FIPS 203/204/205
- NIST PQC project status (2026-07-21)
- NIST IR 8545 — HQC selection
- NIST IR 8610 — Additional Signatures Round 3 (2026)
- NIST SP 800-227 — KEM guidance status
- RFC 5869 — HKDF

Status dəyişkəndir; source və date birlikdə saxlanılır.
